# Level 9: Advanced patterns

## 9.1 A decorator that works with and without arguments

We want both `@log` and `@log(level="WARN")` to work. The trick: if the first argument is the function, we were used without parentheses.

In [ ]:
import functools
import time
import random

# Helpers from earlier notebooks
def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        arg_parts = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        print(f"-> {func.__name__}({', '.join(arg_parts)})")
        result = func(*args, **kwargs)
        print(f"<- {func.__name__} returned {result!r}")
        return result
    return wrapper

class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)

In [ ]:
def log(func=None, *, level="INFO"):
    def decorator(f):
        @functools.wraps(f)
        def wrapper(*args, **kwargs):
            print(f"[{level}] {f.__name__} called")
            return f(*args, **kwargs)
        return wrapper

    if func is None:          # used as @log(...)
        return decorator
    return decorator(func)    # used as @log

@log
def a():
    return 1

@log(level="WARN")
def b():
    return 2

a(); b()

## 9.2 Retry with back-off

Very common in real projects for network calls, APIs and databases.

In [ ]:
import random

def retry(times=3, delay=0.1, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    print(f"attempt {attempt} failed: {e}")
                    if attempt == times:
                        raise
                    time.sleep(delay * attempt)   # wait a bit longer each time
        return wrapper
    return decorator

random.seed(3)

@retry(times=5, delay=0.05, exceptions=(ConnectionError,))
def flaky_api():
    if random.random() < 0.6:
        raise ConnectionError("server busy")
    return "data received"

print(flaky_api())

## 9.3 Registry pattern: decorators that register instead of wrap

A decorator does not have to wrap. It can simply record the function and return it unchanged. Frameworks like Flask (`@app.route`) and pytest (`@pytest.fixture`) use this idea.

In [ ]:
COMMANDS = {}

def command(name):
    def decorator(func):
        COMMANDS[name] = func
        return func          # return the original, unchanged
    return decorator

@command("hello")
def cmd_hello():
    return "Hi there!"

@command("time")
def cmd_time():
    return time.strftime("%H:%M:%S")

for name in ["hello", "time", "unknown"]:
    handler = COMMANDS.get(name)
    print(name, "->", handler() if handler else "command not found")

## 9.4 Argument validation using `inspect`

`inspect.signature` lets the decorator understand the function's parameters, so we can check types from the annotations.

In [ ]:
import inspect

def validate_types(func):
    sig = inspect.signature(func)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()
        for name, value in bound.arguments.items():
            expected = func.__annotations__.get(name)
            if expected is not None and not isinstance(value, expected):
                raise TypeError(
                    f"{name} must be {expected.__name__}, got {type(value).__name__}"
                )
        return func(*args, **kwargs)
    return wrapper

@validate_types
def create_user(name: str, age: int, active: bool = True):
    return {"name": name, "age": age, "active": active}

print(create_user("Anant", 30))
try:
    create_user("Anant", "thirty")
except TypeError as e:
    print("Error:", e)

## 9.5 Decorating classes

A decorator can receive a **class** instead of a function. Here are two examples: adding a method, and the singleton pattern.

In [ ]:
def add_repr(cls):
    def __repr__(self):
        fields = ", ".join(f"{k}={v!r}" for k, v in vars(self).items())
        return f"{cls.__name__}({fields})"
    cls.__repr__ = __repr__
    return cls

@add_repr
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

print(Point(3, 4))

In [ ]:
def singleton(cls):
    instances = {}
    @functools.wraps(cls)
    def get_instance(*args, **kwargs):
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]
    return get_instance

@singleton
class Config:
    def __init__(self):
        print("Loading config...")
        self.debug = True

c1 = Config()
c2 = Config()
print(c1 is c2)

## 9.6 Decorating methods

Function-based decorators work on methods without changes, because `self` just arrives as the first item in `*args`.

In [ ]:
class BankAccount:
    def __init__(self, balance):
        self.balance = balance

    @log_calls
    def withdraw(self, amount):
        self.balance -= amount
        return self.balance

acc = BankAccount(100)
acc.withdraw(30)

> **Gotcha:** the class-based `CountCalls` from Level 7 does **not** work on methods as written, because a plain class instance is not a descriptor, so `self` is never passed in. Fix it by adding a `__get__` method, or use a function-based decorator. The cell below shows the problem.

In [ ]:
class Greeter:
    @CountCalls
    def hi(self):
        return "hi"

try:
    Greeter().hi()
except TypeError as e:
    print("Error:", e)

In [ ]:
# The fix: implement __get__ so the decorator binds to the instance like a normal method
class CountCallsMethodSafe(CountCalls):
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return functools.partial(self.__call__, instance)

class Greeter:
    @CountCallsMethodSafe
    def hi(self):
        return "hi"

g = Greeter()
print(g.hi(), g.hi())
print("calls:", Greeter.hi.calls)

## 9.7 Decorating async functions

In async code the wrapper must also be `async` and must `await` the original.

In [ ]:
import asyncio

def async_timer(func):
    @functools.wraps(func)
    async def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = await func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.3f}s")
        return result
    return wrapper

@async_timer
async def fetch_data():
    await asyncio.sleep(0.2)
    return {"status": "ok"}

# Jupyter already runs an event loop, so we can await at the top level:
await fetch_data()